In [1]:
# 1 - LIMPEZA E PRE-PROCESSAMENTO

import os
import pandas as pd
import mne
from mne_icalabel import label_components

# Configurações de diretório
raw_dir = "../data/raw/"
proc_dir = "../data/processed/"

# Certifique-se de que a pasta processed existe
os.makedirs(proc_dir, exist_ok=True)

# Lista de arquivos
arquivos = [f for f in os.listdir(raw_dir) if f.endswith('.csv')]

def processar_um_arquivo(nome_arquivo):
    print(f"--- Iniciando processamento: {nome_arquivo} ---")
    
    # 1. Carregar e converter
    df = pd.read_csv(os.path.join(raw_dir, nome_arquivo))
    # Selecionando as colunas de canais EEG
    canais_eeg = ['FP1', 'F7', 'F3', 'T7', 'C3', 'P7', 'P3', 'O1', 
                  'O2', 'P4', 'P8', 'C4', 'T8', 'F4', 'F8', 'FP2']
    data = df[canais_eeg].values.T * 1e-6
    
    info = mne.create_info(ch_names=canais_eeg, sfreq=512, ch_types='eeg')
    raw = mne.io.RawArray(data, info)
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'), match_case=False)
    
    # 2. Pré-processamento: Filtros
    raw.notch_filter(60)
    raw.filter(1.0, 45.0)
    
    # 3. ICA e Limpeza via ICLabel
    ica = mne.preprocessing.ICA(n_components=16, random_state=97, method='fastica')
    ica.fit(raw)
    ic_labels = label_components(raw, ica, method='iclabel')
    
    # Exclusão conservadora (Manter apenas o que for Brain > 80%)
    labels = ic_labels["labels"]
    probs = ic_labels["y_pred_proba"]
    exclude = [i for i, (l, p) in enumerate(zip(labels, probs)) if not (l == 'brain' and p >= 0.8)]
    ica.exclude = exclude
    ica.apply(raw)
    
    # 4. Corte em 10 minutos (600 segundos)
    raw.crop(tmin=0, tmax=600)
    
    # 5. Salvar em formato .fif
    output_name = nome_arquivo.replace(".csv", "_cleaned.fif")
    raw.save(os.path.join(proc_dir, output_name), overwrite=True)
    print(f"Arquivo salvo com sucesso: {output_name}\n")

# Loop para processar tudo
for arq in arquivos:
    try:
        processar_um_arquivo(arq)
    except Exception as e:
        print(f"Erro ao processar {arq}: {e}")

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Iniciando processamento: record-[2025.08.12-10.01.51].csv ---
Creating RawArray with float64 data, n_channels=16, n_times=466640
    Range : 0 ... 466639 =      0.000 ...   911.404 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 3381 samples (6.604 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method


/tmp/ipykernel_57713/2622489667.py:39: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')
/tmp/ipykernel_57713/2622489667.py:39: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')
/tmp/ipykernel_57713/2622489667.py:39: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designed with extended infomax ICA decompositions. To use the extended infomax algorithm, use the 'mne.preprocessing.ICA' instance with the arguments 'I

Applying ICA to Raw instance
    Transforming to ICA space (16 components)
    Zeroing out 8 ICA components
    Projecting back using 16 PCA components
Writing /workspaces/eeg-neuromodulation-mdd/notebooks/../data/processed/record-[2025.08.12-10.01.51]_cleaned.fif


/tmp/ipykernel_57713/2622489667.py:53: RuntimeWarning: This filename (/workspaces/eeg-neuromodulation-mdd/notebooks/../data/processed/record-[2025.08.12-10.01.51]_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(os.path.join(proc_dir, output_name), overwrite=True)


...

Arquivo salvo com sucesso: record-[2025.10.20-09.47.01]_cleaned.fif

--- Iniciando processamento: record-[2025.10.23-15.12.30].csv ---
Creating RawArray with float64 data, n_channels=16, n_times=464816
    Range : 0 ... 464815 =      0.000 ...   907.842 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 3381 samples (6.604 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-c

/tmp/ipykernel_57713/2622489667.py:39: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')
/tmp/ipykernel_57713/2622489667.py:39: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')
/tmp/ipykernel_57713/2622489667.py:39: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designed with extended infomax ICA decompositions. To use the extended infomax algorithm, use the 'mne.preprocessing.ICA' instance with the arguments 'I

Applying ICA to Raw instance
    Transforming to ICA space (16 components)
    Zeroing out 7 ICA components
    Projecting back using 16 PCA components
Writing /workspaces/eeg-neuromodulation-mdd/notebooks/../data/processed/record-[2025.10.23-15.12.30]_cleaned.fif


/tmp/ipykernel_57713/2622489667.py:53: RuntimeWarning: This filename (/workspaces/eeg-neuromodulation-mdd/notebooks/../data/processed/record-[2025.10.23-15.12.30]_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(os.path.join(proc_dir, output_name), overwrite=True)


Closing /workspaces/eeg-neuromodulation-mdd/notebooks/../data/processed/record-[2025.10.23-15.12.30]_cleaned.fif
[done]
Arquivo salvo com sucesso: record-[2025.10.23-15.12.30]_cleaned.fif

